In [ ]:
# Cell 1 — Setup
# Requires:
#   pip install -r requirements.txt
#   python -m spacy download en_core_web_sm
# Launch Jupyter from the repo root:
#   jupyter lab notebooks/pipeline_demo.ipynb

import sys
from pathlib import Path

# REPO_ROOT: assumes the notebook kernel was started from the repo root.
# If not, set REPO_ROOT manually, e.g.:
#   REPO_ROOT = Path(r'C:/Users/marta/Documents/PROJECT BARCELONA/GRAMMAR')
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'grammar_parser').is_dir():
    raise RuntimeError(
        f"grammar_parser/ not found under {REPO_ROOT}.\n"
        "Set REPO_ROOT manually at the top of this cell."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import spacy
from grammar_parser import Group1Parser

nlp = spacy.load('en_core_web_sm')

parser1 = Group1Parser(nlp)

# To add future parsers: append them here.
# e.g. parsers = [parser1, Group2Parser(nlp), Group3Parser(nlp)]
parsers = [parser1]

print(f"spaCy model : {nlp.meta['name']} v{nlp.meta['version']}")
print(f"Group1Parser: {len(parser1.structures)} structures loaded")
print(f"Active parsers: {len(parsers)}")

In [ ]:
# Cell 2 — Load lesson sentences

LESSON_PATH = REPO_ROOT / 'processed_data' / 'sentences' / 'Student-1' / 'lesson-1' / 'merged.txt'

def load_sentences(path: Path) -> list:
    with path.open(encoding='utf-8') as fh:
        lines = [line.strip() for line in fh]
    return [line for line in lines if line]

sentences = load_sentences(LESSON_PATH)

print(f"Loaded {len(sentences)} sentences from:")
print(f"  {LESSON_PATH}")
print()
print('First 5 sentences:')
for s in sentences[:5]:
    print(f'  {s!r}')

In [ ]:
# Cell 3 — Pipeline function
# run_pipeline accepts any list of parser objects that implement .parse(doc).
# Adding Group2Parser later: parsers = [parser1, parser2]

def run_pipeline(sentences, nlp, parsers):
    """
    Process all sentences through all parsers.

    Returns
    -------
    list of dict:
        sentence_index : int
        sentence_text  : str
        matches        : list[dict]  — aggregated results from all parsers
    """
    results = []
    # nlp.pipe() batches tokenisation — faster than nlp(s) in a loop.
    docs = list(nlp.pipe(sentences))

    for idx, (sentence, doc) in enumerate(zip(sentences, docs)):
        all_matches = []
        for parser in parsers:
            all_matches.extend(parser.parse(doc))
        results.append({
            'sentence_index': idx,
            'sentence_text': sentence,
            'matches': all_matches,
        })

    return results

print('run_pipeline defined.')

In [ ]:
# Cell 4 — Run the pipeline

pipeline_results = run_pipeline(sentences, nlp, parsers)

total_matches = sum(len(r['matches']) for r in pipeline_results)
sentences_with_hits = sum(1 for r in pipeline_results if r['matches'])

print('Pipeline complete.')
print(f'  Sentences processed  : {len(pipeline_results)}')
print(f'  Sentences with hits  : {sentences_with_hits}')
print(f'  Total match events   : {total_matches}')

In [ ]:
# Cell 5 — Human-readable display
#
# Only shows sentences that have at least one match.
#
# Matches are grouped by token span: all structures that fired on the same
# token range are shown under one SPAN header. This prevents the ~144
# MODALITY structures that share the same modal-verb pattern from printing
# 144 separate lines for a single 'can'.

from collections import defaultdict

def display_results(pipeline_results):
    sentences_shown = 0

    for result in pipeline_results:
        matches = result['matches']
        if not matches:
            continue

        sentences_shown += 1
        idx = result['sentence_index']
        text = result['sentence_text']
        print(f"\n[{idx:03d}] {text}")
        print('      ' + '─' * min(len(text), 80))

        # Group by (start_token, end_token, span_text)
        span_groups = defaultdict(list)
        for m in matches:
            key = (m['start_token'], m['end_token'], m['span_text'])
            span_groups[key].append(m)

        # Sort spans by position in sentence
        for (start, end, span_text), group in sorted(span_groups.items()):
            # Sort structures within a span by CEFR level
            group_sorted = sorted(group, key=lambda m: m['lowest_level_numeric'])
            print(f"      SPAN '{span_text}' (tokens {start}–{end-1}):")
            for m in group_sorted:
                print(
                    f"        [{m['category']:17s}] "
                    f"{m['guideword']:<50s} "
                    f"level={m['lowest_level']}"
                )

    print(f"\n{'─'*80}")
    print(f"Displayed {sentences_shown} / {len(pipeline_results)} sentences with matches.")


display_results(pipeline_results)

In [ ]:
# Cell 6 — CEFR level & category distribution summary

from collections import Counter

def level_summary(pipeline_results):
    level_counts = Counter()
    category_counts = Counter()

    for result in pipeline_results:
        for m in result['matches']:
            level_counts[m['lowest_level']] += 1
            category_counts[m['category']] += 1

    print('CEFR Level Distribution (by lowest_level of matched structure):')
    for level in ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']:
        count = level_counts[level]
        bar = '█' * (count // 10)
        print(f'  {level}: {count:5d}  {bar}')

    print()
    print('Category Distribution:')
    for cat, count in category_counts.most_common():
        print(f'  {cat:22s}: {count}')


level_summary(pipeline_results)